# Modelado de datos

Esta notebook entrena y evalúa un modelo predictivo sobre el dataset previamente preparado.

Se implementa un pipeline de Spark que incluye:

- Codificación de variables categóricas (país)
- Ensamblado de features
- Escalado de variables
- Entrenamiento de un modelo de regresión lineal

El flujo asegura consistencia entre entrenamiento e inferencia y evita data leakage.

### Imports

In [1]:
# Core
from pyspark.sql import SparkSession

# Utils
from pyspark.sql import functions as F
import os

# ML
from pyspark.ml.regression import LinearRegression, LinearRegressionModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

Sesion de spark

In [2]:
spark = SparkSession.builder \
    .appName("penguin-modeling") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/16 11:25:58 WARN Utils: Your hostname, Runa, resolves to a loopback address: 127.0.1.1; using 192.168.0.7 instead (on interface wlp58s0)
26/04/16 11:25:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 11:25:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Parametros

TEST_MONTHS = 3 # meses tomados para conjunto de test

## Lectura de datos

In [4]:
path_actual = os.getcwd()
path_processed = os.path.join(path_actual, "..", "data", "processed", "data.parquet")

df = spark.read.parquet(path_processed)

df.show(5)

+-------+----------+---------------+------------+---------------+-----+-------+------------+----+-----+-----------+----------+----------+-------------+-------+-------+
|country|     fecha|marketing_spend|discount_pct|stock_available|price|  sales|fecha_parsed|year|month|day_of_week|is_weekend|is_holiday|is_non_labour|  lag_1|  lag_7|
+-------+----------+---------------+------------+---------------+-----+-------+------------+----+-----+-----------+----------+----------+-------------+-------+-------+
|     AR|01/04/2023|           4112|       0.069|            341|38.69|3755.44|  2023-04-01|2023|    4|          7|         1|         0|            1|3160.28|2031.51|
|     AR|01/05/2023|           2978|       0.183|           1025|19.62| 4620.3|  2023-05-01|2023|    5|          2|         0|         1|            1|3755.44|4387.11|
|     AR|01/05/2023|           5443|       0.227|            639|27.77|5362.36|  2023-05-01|2023|    5|          2|         0|         1|            1| 4620.3|2

guardado de fecha máxima dataset:

In [5]:
max_fecha = df.select(F.max("fecha_parsed")).collect()[0][0]

## Modelado

### Train test split

In [6]:
split_date = F.add_months(F.lit(max_fecha), - TEST_MONTHS)

train_df = df.filter(F.col("fecha_parsed") < split_date)
test_df = df.filter(F.col("fecha_parsed") >= split_date)

# Label
train_df = train_df.withColumn("label", F.col("sales"))
test_df = test_df.withColumn("label", F.col("sales"))

Se guarda columna lag_7 como predicción baseline

In [7]:
naive_df = test_df.withColumn(
    "naive_prediction",
    F.col("lag_7")
)

### Stages Pipeline

In [8]:
# indexar categorias
indexer = StringIndexer(
    inputCol="country",
    outputCol="country_index",
    handleInvalid="keep"
)

# codificar one-hot (no orden)
encoder = OneHotEncoder(
    inputCols=["country_index"],
    outputCols=["country_ohe"],
    handleInvalid="keep",
    dropLast=True
)

### Selección Features

Se seleccionan variables que capturan:

- Factores de negocio: precio, descuento, marketing, stock
- Estacionalidad: mes, día de la semana
- Calendario: fines de semana y feriados
- Dependencia temporal: lag de ventas
- Diferencias regionales: país

In [9]:
feature_cols = [
    "marketing_spend",
    "discount_pct",
    "stock_available",
    "price",
    "month",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "is_non_labour",
    "lag_7",
    "country_ohe"
]

In [10]:
# Vector Assembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [11]:
# Escalado
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

### Modelo

In [12]:
lr = LinearRegression(
    featuresCol="scaled_features",
    labelCol="label",
    regParam=0.1
)

### Pipeline

In [13]:
pipeline = Pipeline(stages=[
    indexer,
    encoder,
    assembler,
    scaler,
    lr
])

### Entrenamiento

In [14]:
model = pipeline.fit(train_df)

### Predicciones

In [15]:
predictions = model.transform(test_df)

In [16]:
# Mantener columnas útiles
predictions.select("country", "label", "prediction").show(5)

+-------+-------+-----------------+
|country|  label|       prediction|
+-------+-------+-----------------+
|     AR|2635.98|2924.874086641983|
|     AR|3891.68|3839.681774794114|
|     AR|4618.65|4762.304297109917|
|     AR|5976.37|6242.819646854879|
|     AR|5293.84|5439.859415669049|
+-------+-------+-----------------+
only showing top 5 rows


Se utiliza regresión lineal por su simplicidad y facilidad de implementación.

Los resultados reflejan relaciones esperadas entre variables, como el impacto 
negativo del precio y positivo de marketing y descuentos.

Dado el uso de escalado, la magnitud de los coeficientes no es directamente 
interpretable en la escala original. El foco está en la consistencia del pipeline 
y la capacidad predictiva del modelo.

### Evaluación

El modelo se evalúa utilizando métricas estándar de regresión (RMSE, MAE, MAPE) 
y se compara contra un baseline naive basado en valores rezagados.

Los resultados muestran una mejora significativa respecto al baseline, 
indicando que el modelo captura patrones relevantes en los datos.

#### RMSE

Error cuadrático promedio.

In [17]:
evaluator = RegressionEvaluator(
    labelCol="sales",
    predictionCol="naive_prediction",
    metricName="rmse"
)

rmse_naive = evaluator.evaluate(naive_df)
print(f"Naive RMSE: {rmse_naive}")

Naive RMSE: 2065.8628153903396


In [18]:
evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(predictions)

print(f"RMSE: {rmse}")

RMSE: 352.52529981860613


El modelo mejora significativamente respecto al baseline naive.

Mientras que el baseline presenta un RMSE de ~2066, el modelo reduce el error a ~353, 
lo que indica una mejora sustancial en la capacidad predictiva.

#### MAE

El error medio absoluto.

In [19]:
evaluator_mae = RegressionEvaluator(
    labelCol="sales",
    predictionCol="naive_prediction",
    metricName="mae"
)

mae_model = evaluator_mae.evaluate(naive_df)

print(f"Naive MAE: {mae_model}")

Naive MAE: 1742.0737499999998


In [20]:
evaluator_mae = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mae"
)

mae_model = evaluator_mae.evaluate(predictions)

print(f"MAE: {mae_model}")

MAE: 295.0911868463095


También se refleja mejora respecto al baseline naive.

#### MAPE

El MAPE permite interpretar el error en términos porcentuales. 

In [21]:
mape = predictions.select(
    F.mean(F.abs((F.col("label") - F.col("prediction")) / F.col("label")))
).collect()[0][0]

print(f"MAPE: {mape:.4f}")

MAPE: 0.0882


En este caso, el modelo presenta un error promedio del 8.8%.

Se desagrega la diferencia absoluta MAE por paises para percibir mejor el desempeño sectorizado del modelo.

In [22]:
predictions.groupBy("country").agg(
    F.mean(F.abs(F.col("label") - F.col("prediction"))).alias("MAE")
).show()

+-------+------------------+
|country|               MAE|
+-------+------------------+
|     CL|369.27716748565587|
|     UY| 285.7340526770695|
|     AR| 230.2623403762031|
+-------+------------------+



Se observa variabilidad en el error según el país, lo que sugiere que el comportamiento de ventas 
difiere entre mercados. Esto podría justificar, en una versión más avanzada, el entrenamiento de modelos separados por país.

### Interpretación

Coeficientes de las variables

In [23]:
lr_model = [s for s in model.stages if isinstance(s, LinearRegressionModel)][0]

assembler = [s for s in model.stages if isinstance(s, VectorAssembler)][0]

# Nombres de features originales
input_cols = assembler.getInputCols()

feature_names = []

for col_name in input_cols:
    if col_name == "country_ohe":
        sample = predictions.select("country_ohe").head()[0]
        size = len(sample)
        feature_names.extend([f"{col_name}_{i}" for i in range(size)])
    else:
        feature_names.append(col_name)

# Coeficientes
coefficients = lr_model.coefficients.toArray()

feature_importance = sorted(
    zip(feature_names, coefficients),
    key=lambda x: abs(x[1]),
    reverse=True
)

for name, coef in feature_importance:
    print(f"{name}: {coef:.4f}")

marketing_spend: 1092.8581
stock_available: 490.8616
price: -265.3354
month: 243.0002
country_ohe_0: 119.6984
discount_pct: 107.5285
country_ohe_2: -102.3391
lag_7: 43.8859
day_of_week: -33.3455
country_ohe_1: -17.3593
is_holiday: 11.3751
is_weekend: -8.0090
is_non_labour: -3.3325
country_ohe_3: 0.0000


Los coeficientes indican la importancia relativa de las variables. Marketing, stock y precio aparecen como los factores más influyentes, junto con efectos de estacionalidad, país y descuento. Dado el uso de escalado, los valores no son directamente interpretables en unidades originales.

### Guardado modelo

In [24]:
path_model = os.path.join(path_actual, "..", "model", "sales_model")
model.write().overwrite().save(path_model)

26/04/18 09:13:19 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

## Conclusión

Se construyó un modelo simple y consistente que mejora significativamente 
respecto a un baseline naive.

La solución es reproducible, extensible y adecuada como punto de partida 
para modelos más avanzados.
